In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [3]:
REPO_ROOT = Path().resolve().parent

csv_path = REPO_ROOT / 'outputs' / 'period_comparison_all_markets.csv'

df = pd.read_csv(csv_path)

# - High level performance summary

#### - Goal is to show the extent to which performance is changing by Performance Flag, by market and the root causes.

#### Conclusions & Observations
#### 1 - In all 3 markets, each is exhibiting a decline in performance across the comparative time periods. 
#### 2 - For the majority of declines, this is explained as traffic loss brought on by a ranking decline.
#### 3 - Interestingly, a portion of circa 25% of the declining performance can be accounted for by a traffic loss but with no decline in position. This warrants further investigation to understand the reasons for the decline if the data shows no ranking change.

In [4]:
# Creating a dataframe of root causes for performance. Grouping and aggregating in preparation for visualisation

df_root_cause = df.groupby(['market','trend','performance_flag'], as_index=False)['keyword'].count().sort_values('keyword', ascending=False)

In [5]:
# Visualisation show market specific counts of performance flag and trend data.

fig = px.bar(
    df_root_cause,
    x="market",
    y="keyword",
    color="performance_flag",
    facet_col="trend",
    labels={"keyword": "Keyword Count", "market": "Market", "performance_flag": "Performance Flag"},
    title="Keyword Count by Market, Trend, and Performance Flag",
    height=500,
    text_auto=True
)

fig.update_layout(
    barmode="stack",
    legend_title_text="Performance Flag"
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

# - Performance summary broken down by product category levels

### - The aim is to go a bit deeper into what has driven performance changes and draw conclusions about where effort should be prioritised to address the declines.

### Possible Conclusions: Looking at the graph below, the following conclusions could be relevant:
#### 1 - A majority of the declines have been seen in the Snowsports category
#### 2 - Of the snowsports category declines, there is an even split between where traffic lossses have come alongside a ranking decline.
#### 3 - Also within the Snowsports category, a number of declines in clicks have been seen but that has not aligned to a ranking decline. Other factors are likely driving a decline and require further investigation. Such factors could include: ranking cannibalisation, switch in search engine results page features, impacts of technical indexability or crawlability of pages etc.

In [9]:
df[['category_1', 'category_2']] = df['page'].str.split('/', expand=True)[[2, 3]]

In [ ]:
product_categories = df.groupby(['market','trend','performance_flag','category_1','category_2'], as_index=False)['keyword'].count().sort_values('keyword', ascending=False)

product_categories_decline =  product_categories[product_categories['trend'].isin(['Decline', 'Strong Decline'])]

In [21]:
fig = px.bar(
    product_categories_decline,
    x="market",
    y="keyword",
    color="performance_flag",
    facet_col="category_1",
    labels={"keyword": "Keyword Count", "market": "Market", "performance_flag": "Performance Flag"},
    title="Product categories with declining performance by Market and Performance Flag",
    height=500,
    text_auto=True
)

fig.update_layout(
    barmode="stack",
    legend_title_text="Performance Flag"
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()